# Proyección híbrida de cosecha: Prophet por grupo + corrección global con XGBoost

## Propósito del cuaderno
Este cuaderno implementa un enfoque híbrido para la proyección de cosecha diaria por grupo de **Variedad-Bloque**. El procedimiento combina dos niveles de modelado:

1. **Modelado base por grupo con Prophet**, aprovechando la dinámica temporal propia de cada serie.
2. **Corrección global de residuales con XGBoost**, utilizando un panel consolidado de todas las series válidas para mejorar la predicción final.

## Objetivos del flujo
- Cargar y validar la base de datos de entrada.
- Detectar automáticamente las columnas clave requeridas por el proceso.
- Preparar series diarias por grupo para ajuste con Prophet.
- Construir un panel global de residuales históricos.
- Entrenar un modelo XGBoost global sobre esos residuales.
- Combinar ambas salidas mediante blending.
- Generar la predicción para **t+1**.
- Exportar resultados agregados y detallados a Excel.

## Estructura general
1. Importación de librerías y carga de la base.
2. Configuración global del flujo.
3. Definición de funciones auxiliares.
4. Carga automática de hiperparámetros desde archivos JSON.
5. Validación y limpieza de la base.
6. Preparación diaria de cada grupo para Prophet.
7. Ajuste de Prophet por grupo.
8. Construcción del panel global de residuales.
9. Entrenamiento del XGBoost global.
10. Generación de la predicción final.
11. Resúmenes Rojo/Color y exportación.

## Entradas
- Archivo Excel con la base histórica del proyecto.
- Carpeta de hiperparámetros históricos (`HIPERP`), si existe.

## Salidas
- Predicción t+1 por grupo.
- Resúmenes por sector y por gama.
- Totales Rojo/Color.
- Archivo Excel consolidado con resultados.

In [36]:
# =============================================================================
# 0. IMPORTACIÓN DE LIBRERÍAS Y CARGA DE LA BASE DE DATOS
# =============================================================================

# -----------------------------------------------------------------------------
# Librerías estándar
# -----------------------------------------------------------------------------
import os
import json
import re
import logging
import warnings
from pathlib import Path
from typing import Optional, List

# -----------------------------------------------------------------------------
# Librerías para manejo de datos
# -----------------------------------------------------------------------------
import numpy as np
import pandas as pd

# -----------------------------------------------------------------------------
# Configuración general de advertencias
# -----------------------------------------------------------------------------
warnings.filterwarnings("ignore")

# -----------------------------------------------------------------------------
# Mitigar logs de Prophet / cmdstanpy
# -----------------------------------------------------------------------------
os.environ.setdefault("CMDSTANPY_TEMP_DIR", r"C:\Temp\cmdstanpy")
os.makedirs(os.environ["CMDSTANPY_TEMP_DIR"], exist_ok=True)
logging.getLogger("cmdstanpy").setLevel(logging.CRITICAL)

# -----------------------------------------------------------------------------
# Import de Prophet con compatibilidad
# -----------------------------------------------------------------------------
try:
    from prophet import Prophet
except Exception:
    from fbprophet import Prophet

# -----------------------------------------------------------------------------
# Import de XGBoost con validación de disponibilidad
# -----------------------------------------------------------------------------
try:
    from xgboost import XGBRegressor
    XGB_OK = True
except Exception:
    XGB_OK = False
    print("⚠️ No encuentro xgboost. Se entrenará SOLO Prophet.")

# -----------------------------------------------------------------------------
# Carga de la base principal
# -----------------------------------------------------------------------------
ruta_archivo = r'C:\Users\JuanPabloRuiz\OneDrive - C.I.SUNSHINE BOUQUET SAS\Documentos\Base Proyecto Proyeccion Ofrecimiento23-04.xlsx'

# Lectura del archivo Excel en el DataFrame principal del flujo
df = pd.read_excel(ruta_archivo)

# Vista rápida para comprobar la carga
df

,Finca,Variedad,Bloque,Sector,Gama,Fecha Cosecha,Cosecha,Alineamiento,Cantidad (cosecha+Aline) Herramienta Gestion,Edad Variedad(Años),...,Radiación Solar W/m2(DLI)-2,Radiación Solar W/m2(DLI)-3,Grados día,Grados día R-1,Grados día R-2,Grados día R-3,Precip_,Precip_ R-1,Precip_ R-2,Precip_ R-3
0,BETANIA,SWEETNESS,12,SECTOR 2,CREAM,2021-01-04,437,0,546,16.0,...,0.00000,0.00000,3.482708,0.0,0.00,0.00,13.860000,0.0,0.0,0.0
1,BETANIA,SWEETNESS,44,EL FUERTE,CREAM,2021-01-04,75,0,176,18.0,...,0.00000,0.00000,3.482708,0.0,0.00,0.00,12.687292,0.0,0.0,0.0
2,BETANIA,SKYLINE,16,SECTOR 2,FALL,2021-01-04,963,492,1058,25.0,...,0.00000,0.00000,3.482708,0.0,0.00,0.00,12.687292,0.0,0.0,0.0
3,BETANIA,HIGH & MAGIC,3,SECTOR 1,FALL,2021-01-04,150,0,180,18.0,...,0.00000,0.00000,3.482708,0.0,0.00,0.00,12.687292,0.0,0.0,0.0
4,BETANIA,ORANGE CRUSH,3,SECTOR 1,FALL,2021-01-04,0,0,2515,12.0,...,0.00000,0.00000,3.482708,0.0,0.00,0.00,12.687292,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
233614,BETANIA,FREEDOM,J2,JUAYCA,RED,2026-04-23,9480,0,17071,8.0,...,20.85721,19.39163,8.450000,9.2,8.14,8.29,0.800000,0.0,2.8,0.2
233615,BETANIA,FREEDOM,J3,JUAYCA,RED,2026-04-23,2970,0,4980,8.0,...,20.85721,19.39163,8.450000,9.2,8.14,8.29,0.800000,0.0,2.8,0.2
233616,BETANIA,FREEDOM,J5,JUAYCA,RED,2026-04-23,9780,0,12559,8.0,...,20.85721,19.39163,8.450000,9.2,8.14,8.29,0.800000,0.0,2.8,0.2
233617,BETANIA,CUPIDO,J7,JUAYCA,RED,2026-04-23,1708,0,4028,8.0,...,20.85721,19.39163,8.450000,9.2,8.14,8.29,0.800000,0.0,2.8,0.2


## Configuración general del flujo

En esta sección se definen los parámetros globales del proceso:

- nombres de columnas clave,
- candidatos para detección automática de columnas,
- horizonte de predicción,
- mínimos de datos requeridos,
- estructura de rezagos y medias móviles,
- pesos del blending final,
- y rutas de salida.

También se fijan los hiperparámetros por defecto de Prophet y XGBoost, los cuales podrán ser reemplazados automáticamente si existen archivos JSON válidos en la carpeta de hiperparámetros.

In [37]:
# =============================================================================
# 1) CONFIGURACIÓN GENERAL
# =============================================================================

# Columnas base del modelo
DATE_COL   = "Fecha Cosecha"
TARGET_COL = "Cosecha"
GROUP_COLS = ["Variedad", "Bloque"]

# Posibles nombres de la variable exógena principal
CANT_CANDIDATES = [
    "Cantidad",
    "Cantidad (cosecha+Aline) Herramienta Gestion",
    "Cantidad (cosecha+Aline) Herramienta Gestión",
    "Cantidad (cosecha+Aline)"
]

# Posibles nombres de columnas de metadatos
SECTOR_CANDS = ["Sector", "SECTOR"]
GAMA_CANDS   = ["Gama", "GAMA"]
COLOR_CANDS  = ["Color", "COLOR"]

PLANTS_COL = "Plantas Productivas"
FINCA_CANDS = ["Finca", "FINCA"]
PAIS_CANDS  = ["Pais", "PAIS"]

# Horizonte de predicción
HORIZON_DAYS = 1

# -----------------------------------------------------------------------------
# Mínimos requeridos por grupo
# -----------------------------------------------------------------------------
MIN_NONNA_Y    = 60
MIN_DAYS_DAILY = 120

# -----------------------------------------------------------------------------
# Configuración del modelo global de residuales
# -----------------------------------------------------------------------------
LAGS_USE = [1, 7, 14, 28]
ROLLS    = [7, 28]
TEST_DAYS_RESID = 45
MIN_ROWS_GLOBAL_XGB = 300

# -----------------------------------------------------------------------------
# Pesos de blending final
# -----------------------------------------------------------------------------
BLEND_CORRECTED = 0.70
BLEND_PROPHET   = 0.30

# -----------------------------------------------------------------------------
# Carpeta de exportación
# -----------------------------------------------------------------------------
OUTPUT_DIR = r"C:\Users\JuanPabloRuiz\OneDrive - C.I.SUNSHINE BOUQUET SAS\Documentos\BDENVIO"

# -----------------------------------------------------------------------------
# Carpeta raíz de hiperparámetros
# -----------------------------------------------------------------------------
HIPERP_ROOT = Path(
    r"C:\Users\JuanPabloRuiz\OneDrive - C.I.SUNSHINE BOUQUET SAS\Documentos\HIPERP"
)
PIPELINE_FAMILY = "HIBRIDO"

# =============================================================================
# 1.1) HIPERPARÁMETROS POR DEFECTO
# =============================================================================

# Prophet híbrido por defecto
PROPHET_HYB_DEFAULTS = dict(
    changepoint_prior_scale=0.05,
    seasonality_prior_scale=10.0,
    seasonality_mode="additive",
    yearly_seasonality=True,
    weekly_seasonality=True,
    n_changepoints=15,
    use_cycle90=False,
    use_exog=True,
    use_plants=True
)

# XGBoost de residuales por defecto
XGB_RESID_DEFAULTS = dict(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    min_child_weight=5,
    objective="reg:squarederror",
    n_jobs=-1,
    random_state=42,
    tree_method="hist"
)

## Funciones auxiliares

A continuación se agrupan las funciones que soportan el flujo principal. Estas funciones cumplen tareas como:

- impresión formateada de tablas,
- detección automática de columnas,
- construcción del grupo Rojo/Color,
- limpieza de la base,
- creación de variables temporales,
- construcción de rezagos y medias móviles,
- cálculo de valores fallback,
- y lectura automática de hiperparámetros almacenados en JSON.

Esta organización mejora la legibilidad del script y deja el flujo principal más limpio.

In [38]:
# =============================================================================
# 2) HELPERS GENERALES
# =============================================================================

def safe_int_round(x):
    """
    Redondea un valor a entero asegurando que no sea negativo.
    """
    return int(np.rint(max(float(x), 0.0)))


def pretty_print_df(title: str, dfx: pd.DataFrame, max_rows: int = 60):
    """
    Imprime un DataFrame con formato más legible en consola.
    """
    print("\n" + title)
    print("-" * len(title))
    with pd.option_context(
        "display.max_rows", max_rows,
        "display.max_columns", 200,
        "display.width", 220
    ):
        print(dfx.to_string(index=False))


def detect_col(df_, cands):
    """
    Detecta la primera columna existente dentro de una lista de candidatos.
    También intenta coincidencia parcial si no encuentra un match exacto.
    """
    for c in cands:
        if c in df_.columns:
            return c

    for col in df_.columns:
        for c in cands:
            if str(c).lower() in str(col).lower():
                return col

    return None


def build_color_group(df_):
    """
    Construye el grupo de color (Rojo / Color) a partir de las columnas
    Gama y Color, usando búsqueda por palabras clave.
    """
    idx = df_.index

    gama_u = (
        df_["Gama"].astype(str).str.upper().str.strip()
        if "Gama" in df_.columns else pd.Series("", index=idx)
    )
    color_u = (
        df_["Color"].astype(str).str.upper().str.strip()
        if "Color" in df_.columns else pd.Series("", index=idx)
    )

    is_red = (
        gama_u.str.contains(r"\bRED\b|\BROJO\b", regex=True, na=False)
        | color_u.str.contains(r"\bRED\b|\BROJO\b", regex=True, na=False)
    )

    is_cream = (
        gama_u.str.contains(r"\bCREAM\b|\BCREMA\b", regex=True, na=False)
        | color_u.str.contains(r"\bCREAM\b|\BCREMA\b", regex=True, na=False)
    )

    return np.where(is_red, "Rojo", np.where(is_cream, "Color", "Color"))


def clean_base(df_in: pd.DataFrame) -> pd.DataFrame:
    """
    Limpia la base:
    - convierte fechas,
    - normaliza columnas de grupo,
    - estandariza algunos metadatos,
    - convierte el target a numérico,
    - elimina filas sin información mínima,
    - y excluye fechas futuras.
    """
    out = df_in.copy()
    out[DATE_COL] = pd.to_datetime(out[DATE_COL], errors="coerce", dayfirst=True)

    for c in GROUP_COLS:
        out[c] = out[c].astype(str).str.strip().str.upper()

    for cc in [
        detect_col(out, SECTOR_CANDS),
        detect_col(out, GAMA_CANDS),
        detect_col(out, COLOR_CANDS),
        detect_col(out, FINCA_CANDS),
        detect_col(out, PAIS_CANDS)
    ]:
        if cc and cc in out.columns:
            out[cc] = out[cc].astype(str).str.strip()

    out[TARGET_COL] = pd.to_numeric(out[TARGET_COL], errors="coerce")
    out = out.dropna(subset=[DATE_COL] + GROUP_COLS).copy()

    # Se eliminan fechas futuras respecto al día de ejecución
    today_cutoff = pd.Timestamp.today().normalize()
    out = out[out[DATE_COL] <= today_cutoff].copy()

    out = out.sort_values(GROUP_COLS + [DATE_COL]).reset_index(drop=True)
    return out


def add_time_feats(dts: pd.Series) -> pd.DataFrame:
    """
    Genera variables temporales calendáricas y cíclicas a partir de una serie de fechas.
    """
    dts = pd.to_datetime(dts)
    doy = dts.dt.dayofyear.astype(int)
    woy = dts.dt.isocalendar().week.astype(int)

    return pd.DataFrame({
        "Dia_Sin": np.sin(2*np.pi*doy/365.25),
        "Dia_Cos": np.cos(2*np.pi*doy/365.25),
        "Sem_Sin": np.sin(2*np.pi*woy/52.14),
        "Sem_Cos": np.cos(2*np.pi*woy/52.14),
        "Mes": dts.dt.month.astype(int),
        "DiaSemana": dts.dt.dayofweek.astype(int),
        "EsFinSemana": (dts.dt.dayofweek >= 5).astype(int),
    })


def build_group_lags_rolls(df_, group_cols, col, lags, rolls):
    """
    Construye rezagos y medias móviles por grupo para una columna dada.
    """
    df_ = df_.sort_values(group_cols + ["ds"]).copy()
    g = df_.groupby(group_cols, sort=False)[col]

    for L in lags:
        df_[f"{col}_Lag_{L}"] = g.shift(L)

    for w in rolls:
        df_[f"{col}_RollMean_{w}"] = (
            df_.groupby(group_cols, sort=False)[col]
               .transform(lambda s: s.shift(1).rolling(w, min_periods=2).mean())
        )

    return df_


def fallback_pred(gd: pd.DataFrame) -> float:
    """
    Predicción de respaldo para grupos con poca información:
    - promedio de los últimos 7 días si existe,
    - si no, último valor observado,
    - si no, cero.
    """
    s = gd.set_index("ds")[TARGET_COL].astype(float)
    last7 = s.tail(7)

    if len(last7) and last7.notna().any():
        return float(last7.mean())

    if s.notna().any():
        return float(s.dropna().iloc[-1])

    return 0.0


def resumen_rojo_color(df_in: pd.DataFrame, group_col: str) -> pd.DataFrame:
    """
    Resume las predicciones agregando Rojo, Color y Total por la dimensión solicitada.
    """
    tmp = (
        df_in.groupby([group_col, "ColorGrupo"], dropna=False)["Cosecha_Predicha"]
           .sum()
           .unstack("ColorGrupo", fill_value=0)
    )

    for c in ["Rojo", "Color"]:
        if c not in tmp.columns:
            tmp[c] = 0

    tmp["Total"] = tmp["Rojo"] + tmp["Color"]
    tmp = tmp[["Rojo", "Color", "Total"]].reset_index()

    return tmp.sort_values("Total", ascending=False)


def add_code_column(df_, col):
    """
    Convierte una columna categórica a códigos enteros mediante factorization.
    """
    vals = df_[col].astype(str).fillna("MISSING")
    codes, _ = pd.factorize(vals)
    df_[f"{col}_Code"] = codes.astype("int32")
    return df_

## Funciones para carga automática de hiperparámetros

Estas funciones permiten localizar archivos JSON de hiperparámetros previamente guardados dentro de la carpeta `HIPERP`.

El mecanismo:
- identifica la semana actual,
- busca primero un archivo exacto para esa semana,
- si no existe, busca la versión más reciente disponible,
- y finalmente mezcla esos parámetros con los valores por defecto.

Así, el flujo sigue funcionando incluso si no hay archivos JSON disponibles.

In [39]:
# =============================================================================
# 3) HELPERS PARA HIPERPARÁMETROS
# =============================================================================

def _parse_week_folder(name: str):
    """
    Interpreta carpetas con formato YYYY_WWW.
    """
    m = re.fullmatch(r"(\d{4})_W(\d{2})", str(name))
    if not m:
        return None
    return (int(m.group(1)), int(m.group(2)))


def _current_run_week():
    """
    Devuelve la semana ISO actual en formato YYYY_WWW.
    """
    ts = pd.Timestamp.today().normalize()
    iso = ts.isocalendar()
    return f"{int(iso.year)}_W{int(iso.week):02d}"


def _list_week_dirs(root: Path):
    """
    Lista las carpetas semanales válidas ordenadas de la más reciente a la más antigua.
    """
    if not root.exists():
        return []

    out = []
    for p in root.iterdir():
        if p.is_dir():
            wk = _parse_week_folder(p.name)
            if wk is not None:
                out.append((wk, p))

    out.sort(key=lambda x: x[0], reverse=True)
    return out


def _find_hyperparam_json(model_name: str, family: str = "HIBRIDO") -> Optional[Path]:
    """
    Busca el archivo JSON de hiperparámetros más adecuado para un modelo.
    """
    if not HIPERP_ROOT.exists():
        return None

    run_week = _current_run_week()
    preferred = HIPERP_ROOT / run_week / family / f"best_params_{family}_{model_name}_{run_week}.json"
    if preferred.exists():
        return preferred

    week_dirs = _list_week_dirs(HIPERP_ROOT)
    for _, week_dir in week_dirs:
        cand = week_dir / family / f"best_params_{family}_{model_name}_{week_dir.name}.json"
        if cand.exists():
            return cand

    pattern = f"best_params_{family}_{model_name}_*.json"
    all_matches = sorted(HIPERP_ROOT.glob(f"*/{family}/{pattern}"), reverse=True)
    return all_matches[0] if all_matches else None


def _load_best_params_json(model_name: str, family: str = "HIBRIDO"):
    """
    Carga el contenido de un JSON de hiperparámetros si existe.
    """
    path = _find_hyperparam_json(model_name=model_name, family=family)
    if path is None:
        return None, None

    try:
        with open(path, "r", encoding="utf-8") as f:
            payload = json.load(f)
        best_params = payload.get("best_params", {})
        return best_params, path
    except Exception:
        return None, path


def _merge_params(default_params: dict, tuned_params: Optional[dict], allowed_keys: List[str]):
    """
    Mezcla parámetros por defecto con parámetros ajustados permitidos.
    """
    params = default_params.copy()

    if tuned_params:
        for k in allowed_keys:
            if k in tuned_params:
                params[k] = tuned_params[k]

    return params


def load_all_hybrid_hyperparams():
    """
    Carga los hiperparámetros disponibles para Prophet híbrido y XGB residual híbrido.
    """
    loaded_info = []

    prophet_tuned, prophet_path = _load_best_params_json("PROPHET_HYB", PIPELINE_FAMILY)
    xgb_tuned, xgb_path = _load_best_params_json("XGB_RESID_HYB", PIPELINE_FAMILY) if XGB_OK else (None, None)

    PROPHET_HYB_PARAMS = _merge_params(
        PROPHET_HYB_DEFAULTS,
        prophet_tuned,
        [
            "changepoint_prior_scale",
            "seasonality_prior_scale",
            "seasonality_mode",
            "yearly_seasonality",
            "weekly_seasonality",
            "n_changepoints",
            "use_cycle90",
            "use_exog",
            "use_plants"
        ]
    )

    XGB_RESID_PARAMS = _merge_params(
        XGB_RESID_DEFAULTS,
        xgb_tuned,
        [
            "n_estimators",
            "learning_rate",
            "max_depth",
            "subsample",
            "colsample_bytree",
            "reg_lambda",
            "min_child_weight"
        ]
    ) if XGB_OK else None

    loaded_info.append({
        "Modelo": "PROPHET_HYB",
        "Fuente": str(prophet_path) if prophet_path else "DEFAULT",
        "Usando_json": bool(prophet_tuned)
    })

    if XGB_OK:
        loaded_info.append({
            "Modelo": "XGB_RESID_HYB",
            "Fuente": str(xgb_path) if xgb_path else "DEFAULT",
            "Usando_json": bool(xgb_tuned)
        })

    return PROPHET_HYB_PARAMS, XGB_RESID_PARAMS, pd.DataFrame(loaded_info)


# Carga efectiva de hiperparámetros
PROPHET_HYB_PARAMS, XGB_RESID_PARAMS, df_hyper_loaded = load_all_hybrid_hyperparams()

## Validación de la base y detección de columnas reales

En esta etapa se valida que el DataFrame exista y contenga las columnas obligatorias mínimas.

Luego se:
- limpia la base,
- detectan automáticamente las columnas reales de Cantidad, Sector, Gama, Color, Finca y País,
- identifica la última fecha real disponible de cosecha,
- y se imprime el origen de los hiperparámetros utilizados.

Esta sección asegura que el flujo se adapte a pequeñas variaciones de nombres en la base.

In [40]:
# =============================================================================
# 4) VALIDAR DATAFRAME
# =============================================================================

if "df" not in globals():
    raise ValueError("No encuentro `df` en memoria. Debes tener tu DataFrame como `df`.")

req = set([DATE_COL, TARGET_COL] + GROUP_COLS)
missing = req - set(df.columns)
if missing:
    raise ValueError(f"Faltan columnas requeridas: {missing}")

# =============================================================================
# 5) LIMPIEZA + DETECCIÓN DE COLUMNAS
# =============================================================================

df0 = clean_base(df)

CANT_COL   = detect_col(df0, CANT_CANDIDATES)
SECTOR_COL = detect_col(df0, SECTOR_CANDS)
GAMA_COL   = detect_col(df0, GAMA_CANDS)
COLOR_COL  = detect_col(df0, COLOR_CANDS)
FINCA_COL  = detect_col(df0, FINCA_CANDS)
PAIS_COL   = detect_col(df0, PAIS_CANDS)

if CANT_COL is None:
    raise ValueError(f"No pude detectar la columna de Cantidad. Candidatos: {CANT_CANDIDATES}")

# Última fecha real de cosecha, no simplemente última fecha existente
last_date_global = pd.to_datetime(
    df0.loc[df0[TARGET_COL].notna(), DATE_COL]
).max()

next_date_global = last_date_global + pd.Timedelta(days=HORIZON_DAYS)

# Resumen de detección
print(f"✅ Detectado CANT_COL: {CANT_COL}")
print(f"✅ Detectado SECTOR_COL: {SECTOR_COL}")
print(f"✅ Detectado GAMA_COL: {GAMA_COL}")
print(f"✅ Detectado COLOR_COL: {COLOR_COL}")
print(f"✅ Detectado FINCA_COL: {FINCA_COL}")
print(f"✅ Detectado PAIS_COL: {PAIS_COL}")
print(f"📅 Última fecha real de cosecha: {last_date_global.date()} | t+1: {next_date_global.date()}")

pretty_print_df("📦 Hiperparámetros cargados para HIBRIDO", df_hyper_loaded, max_rows=10)

print("\n🧠 Parámetros Prophet híbrido usados:")
print(PROPHET_HYB_PARAMS)

if XGB_OK:
    print("\n🧠 Parámetros XGB residual híbrido usados:")
    print(XGB_RESID_PARAMS)

✅ Detectado CANT_COL: Cantidad (cosecha+Aline) Herramienta Gestion
✅ Detectado SECTOR_COL: Sector
✅ Detectado GAMA_COL: Gama
✅ Detectado COLOR_COL: None
✅ Detectado FINCA_COL: Finca
✅ Detectado PAIS_COL: None
📅 Última fecha real de cosecha: 2026-04-23 | t+1: 2026-04-24

📦 Hiperparámetros cargados para HIBRIDO
---------------------------------------
       Modelo                                                                                                                                        Fuente  Usando_json
  PROPHET_HYB   C:\Users\JuanPabloRuiz\OneDrive - C.I.SUNSHINE BOUQUET SAS\Documentos\HIPERP\2026_W16\HIBRIDO\best_params_HIBRIDO_PROPHET_HYB_2026_W16.json         True
XGB_RESID_HYB C:\Users\JuanPabloRuiz\OneDrive - C.I.SUNSHINE BOUQUET SAS\Documentos\HIPERP\2026_W16\HIBRIDO\best_params_HIBRIDO_XGB_RESID_HYB_2026_W16.json         True

🧠 Parámetros Prophet híbrido usados:
{'changepoint_prior_scale': 0.026378868447050275, 'seasonality_prior_scale': 5.685528645127449, 'seasona

## Preparación diaria por grupo y ajuste con Prophet

A partir de aquí el flujo trabaja grupo por grupo, donde cada grupo corresponde a una combinación de **Variedad-Bloque**.

Para cada grupo se hace lo siguiente:
- se valida si llega hasta la última fecha global,
- se verifica si tiene suficiente información,
- si no cumple mínimos se genera una predicción fallback,
- si cumple, se construye su serie diaria,
- se ajusta un modelo Prophet,
- y se genera tanto el histórico ajustado como la predicción t+1.

El resultado de esta fase es un panel consolidado con todas las series válidas que luego alimentará el modelo global de residuales.

In [41]:
# =============================================================================
# 6) PREPARAR CADA GRUPO PARA PROPHET
# =============================================================================

def prepare_daily_group(gg: pd.DataFrame):
    """
    Prepara un grupo Variedad-Bloque para modelado diario con Prophet.
    Si no cumple condiciones mínimas, devuelve una salida fallback.
    """
    v = str(gg["Variedad"].iloc[0]).strip().upper()
    b = str(gg["Bloque"].iloc[0]).strip().upper()

    sector = str(gg[SECTOR_COL].iloc[0]).strip() if SECTOR_COL else "SIN_SECTOR"
    gama   = str(gg[GAMA_COL].iloc[0]).strip()   if GAMA_COL else "SIN_GAMA"
    colorv = str(gg[COLOR_COL].iloc[0]).strip()  if COLOR_COL else ""
    finca  = str(gg[FINCA_COL].iloc[0]).strip()  if FINCA_COL else ""
    pais   = str(gg[PAIS_COL].iloc[0]).strip()   if PAIS_COL else ""

    g = gg.sort_values(DATE_COL).copy()
    g[TARGET_COL] = pd.to_numeric(g[TARGET_COL], errors="coerce")
    g[CANT_COL]   = pd.to_numeric(g[CANT_COL], errors="coerce")

    if PLANTS_COL in g.columns:
        g[PLANTS_COL] = pd.to_numeric(g[PLANTS_COL], errors="coerce")

    last_real_date = g.loc[g[TARGET_COL].notna(), DATE_COL].max()

    # El grupo debe llegar hasta la última fecha global de cosecha real
    if pd.isna(last_real_date) or pd.to_datetime(last_real_date) != last_date_global:
        return None, "NOT_CURRENT"

    # Si no hay suficientes observaciones válidas del target, se usa fallback
    if g[TARGET_COL].notna().sum() < MIN_NONNA_Y:
        agg_dict = {TARGET_COL: "sum"}
        gd_fb = g.set_index(DATE_COL).resample("D").agg(agg_dict).reset_index().rename(columns={DATE_COL: "ds"})
        yfb = fallback_pred(gd_fb)

        return {
            "mode": "FALLBACK",
            "result": {
                "Variedad": v, "Bloque": b, "Sector": sector, "Gama": gama, "Color": colorv,
                "Finca": finca, "Pais": pais,
                "Fecha_Prediccion": next_date_global.strftime("%Y-%m-%d"),
                "Metodo": "FALLBACK_poca_data_y",
                "Cosecha_Predicha": safe_int_round(yfb)
            }
        }, None

    # Agregación diaria
    agg_dict = {
        TARGET_COL: "sum",
        CANT_COL: "sum",
    }

    if PLANTS_COL in g.columns:
        agg_dict[PLANTS_COL] = "last"

    gd = (
        g.set_index(DATE_COL)
         .resample("D")
         .agg(agg_dict)
         .reset_index()
         .rename(columns={DATE_COL: "ds"})
    )

    gd[CANT_COL] = gd[CANT_COL].fillna(0)

    if PLANTS_COL in gd.columns:
        gd[PLANTS_COL] = gd[PLANTS_COL].ffill().bfill()

    gd = gd[gd["ds"] <= last_date_global].copy().sort_values("ds").reset_index(drop=True)

    # Si el histórico diario es demasiado corto, se usa fallback
    if len(gd) < MIN_DAYS_DAILY:
        yfb = fallback_pred(gd)
        return {
            "mode": "FALLBACK",
            "result": {
                "Variedad": v, "Bloque": b, "Sector": sector, "Gama": gama, "Color": colorv,
                "Finca": finca, "Pais": pais,
                "Fecha_Prediccion": next_date_global.strftime("%Y-%m-%d"),
                "Metodo": "FALLBACK_pocos_dias_diario",
                "Cosecha_Predicha": safe_int_round(yfb)
            }
        }, None

    meta = {
        "Variedad": v, "Bloque": b, "Sector": sector, "Gama": gama, "Color": colorv,
        "Finca": finca, "Pais": pais
    }

    return {
        "mode": "PROPHET",
        "gd": gd,
        "meta": meta
    }, None


# =============================================================================
# 7) AJUSTE DE PROPHET POR GRUPO Y CONSTRUCCIÓN DEL PANEL
# =============================================================================

def fit_prophet_and_build_panel(group_obj):
    """
    Ajusta Prophet para un grupo y devuelve:
    - panel histórico + futuro con componentes,
    - predicción t+1 del modelo Prophet.
    """
    gd = group_obj["gd"].copy()
    meta = group_obj["meta"]

    # Construcción del histórico de covariables
    hist_cov = gd.copy()
    hist_cov["Cantidad_Lag_1"] = hist_cov[CANT_COL].shift(1)
    hist_cov["Cantidad_Lag_1"] = hist_cov["Cantidad_Lag_1"].fillna(hist_cov[CANT_COL].iloc[0])

    prophet_regressors = []

    if PROPHET_HYB_PARAMS.get("use_exog", True):
        prophet_regressors.append("Cantidad_Lag_1")

    if (
        PLANTS_COL in hist_cov.columns
        and PROPHET_HYB_PARAMS.get("use_exog", True)
        and PROPHET_HYB_PARAMS.get("use_plants", True)
    ):
        hist_cov[PLANTS_COL] = hist_cov[PLANTS_COL].ffill().bfill()
        if hist_cov[PLANTS_COL].nunique(dropna=True) > 1:
            prophet_regressors.append(PLANTS_COL)

    prophet_df = hist_cov[["ds", TARGET_COL] + prophet_regressors].copy().rename(columns={TARGET_COL: "y"})
    prophet_df["y"] = prophet_df["y"].astype(float)

    # Construcción del modelo Prophet
    m = Prophet(
        yearly_seasonality=bool(PROPHET_HYB_PARAMS.get("yearly_seasonality", True)),
        weekly_seasonality=bool(PROPHET_HYB_PARAMS.get("weekly_seasonality", True)),
        daily_seasonality=False,
        seasonality_mode=PROPHET_HYB_PARAMS.get("seasonality_mode", "additive"),
        uncertainty_samples=0,
        n_changepoints=int(PROPHET_HYB_PARAMS.get("n_changepoints", 15)),
        changepoint_prior_scale=float(PROPHET_HYB_PARAMS.get("changepoint_prior_scale", 0.05)),
        seasonality_prior_scale=float(PROPHET_HYB_PARAMS.get("seasonality_prior_scale", 10.0))
    )

    if PROPHET_HYB_PARAMS.get("use_cycle90", False):
        m.add_seasonality(name="cycle90", period=90.0, fourier_order=5)

    for reg in prophet_regressors:
        m.add_regressor(reg)

    try:
        m.fit(prophet_df, algorithm="Newton")
    except TypeError:
        m.fit(prophet_df)

    # Construcción de covariables futuras
    future_cov = gd.copy()
    future_row = future_cov.iloc[-1:].copy()
    future_row["ds"] = next_date_global
    future_row[CANT_COL] = float(future_cov[CANT_COL].iloc[-1])

    if PLANTS_COL in future_cov.columns:
        future_row[PLANTS_COL] = float(future_cov[PLANTS_COL].ffill().bfill().iloc[-1])

    future_row[TARGET_COL] = np.nan

    future_cov = pd.concat([future_cov, future_row], ignore_index=True)
    future_cov = future_cov.sort_values("ds").reset_index(drop=True)
    future_cov["Cantidad_Lag_1"] = future_cov[CANT_COL].shift(1)
    future_cov["Cantidad_Lag_1"] = future_cov["Cantidad_Lag_1"].fillna(future_cov[CANT_COL].iloc[0])

    # Futuro requerido por Prophet
    future = m.make_future_dataframe(periods=HORIZON_DAYS, freq="D")
    keep_cols = ["ds"] + prophet_regressors
    future = future.merge(future_cov[keep_cols], on="ds", how="left")

    for c in prophet_regressors:
        future[c] = future[c].ffill().bfill()

    # Pronóstico
    fcst = m.predict(future)

    # Construcción del panel final por grupo
    panel = future_cov.merge(
        fcst[["ds", "yhat", "trend"] + [c for c in ["weekly", "yearly"] if c in fcst.columns]],
        on="ds",
        how="left"
    ).merge(
        prophet_df[["ds", "y"]],
        on="ds",
        how="left"
    )

    if "weekly" not in panel.columns:
        panel["weekly"] = 0.0
    if "yearly" not in panel.columns:
        panel["yearly"] = 0.0

    panel["yhat_prophet"] = panel["yhat"].astype(float)
    panel.drop(columns=["yhat"], inplace=True)

    panel["is_future"] = (panel["ds"] == next_date_global).astype(int)

    for k, v in meta.items():
        panel[k] = v

    t1_prophet = float(panel.loc[panel["ds"] == next_date_global, "yhat_prophet"].iloc[0])

    return panel, t1_prophet


# =============================================================================
# 8) LOOP DE PROPHET POR GRUPO
# =============================================================================

n_groups = df0.groupby(GROUP_COLS, sort=False).ngroups
print(f"\n🚀 Total grupos Variedad–Bloque: {n_groups}\n")

fallback_rows = []
panels = []
prophet_t1_map = {}
skipped_not_current = 0
prophet_error_count = 0

for i, ((v, b), gg) in enumerate(df0.groupby(GROUP_COLS, sort=False), start=1):
    prep, flag = prepare_daily_group(gg)

    if flag == "NOT_CURRENT":
        skipped_not_current += 1
        if (i % 25) == 0 or i == n_groups:
            print(f"✅ {i}/{n_groups} grupos procesados...")
        continue

    if prep["mode"] == "FALLBACK":
        fallback_rows.append(prep["result"])
        if (i % 25) == 0 or i == n_groups:
            print(f"✅ {i}/{n_groups} grupos procesados...")
        continue

    try:
        panel_g, t1_prophet = fit_prophet_and_build_panel(prep)
        panels.append(panel_g)
        prophet_t1_map[(prep["meta"]["Variedad"], prep["meta"]["Bloque"])] = t1_prophet
    except Exception:
        prophet_error_count += 1
        gd_fb = prep["gd"].copy()
        yfb = fallback_pred(gd_fb)
        meta = prep["meta"]

        fallback_rows.append({
            "Variedad": meta["Variedad"], "Bloque": meta["Bloque"],
            "Sector": meta["Sector"], "Gama": meta["Gama"], "Color": meta["Color"],
            "Finca": meta["Finca"], "Pais": meta["Pais"],
            "Fecha_Prediccion": next_date_global.strftime("%Y-%m-%d"),
            "Metodo": "FALLBACK_Prophet",
            "Cosecha_Predicha": safe_int_round(yfb)
        })

    if (i % 25) == 0 or i == n_groups:
        print(f"✅ {i}/{n_groups} grupos procesados...")

if len(panels) == 0 and len(fallback_rows) == 0:
    raise ValueError("⚠️ No hubo paneles Prophet ni fallbacks. Revisa filtros de fechas y mínimos.")


🚀 Total grupos Variedad–Bloque: 165

✅ 25/165 grupos procesados...
✅ 50/165 grupos procesados...
✅ 75/165 grupos procesados...
✅ 100/165 grupos procesados...
✅ 125/165 grupos procesados...
✅ 150/165 grupos procesados...
✅ 165/165 grupos procesados...


## Construcción del panel global de residuales y corrección con XGBoost

Una vez ajustado Prophet por grupo, se concatenan todos los paneles válidos y se construye un panel global.

Sobre ese panel se calcula:

- el residual histórico `y - yhat_prophet`,
- rezagos y medias móviles del target,
- rezagos y medias móviles de la variable Cantidad,
- variables de tiempo,
- y codificaciones numéricas de algunas variables categóricas.

Con esa matriz se entrena un **XGBoost global de residuales**.  
Luego, para t+1, se genera una predicción corregida que se combina con la salida base de Prophet mediante blending.

In [42]:
# =============================================================================
# 9) PANEL GLOBAL DE RESIDUALES
# =============================================================================

if len(panels) > 0:
    panel_all = pd.concat(panels, ignore_index=True)
else:
    panel_all = pd.DataFrame()

global_xgb_ready = False
xgb_model = None
feature_cols = []

if XGB_OK and len(panel_all) > 0:
    panel_all = panel_all.sort_values(GROUP_COLS + ["ds"]).reset_index(drop=True)

    # Residual histórico de Prophet
    panel_all["resid"] = panel_all["y"] - panel_all["yhat_prophet"]

    # Rezagos y medias móviles del target y la variable exógena principal
    panel_all = build_group_lags_rolls(panel_all, GROUP_COLS, "y", LAGS_USE, ROLLS)
    panel_all = build_group_lags_rolls(panel_all, GROUP_COLS, CANT_COL, LAGS_USE, ROLLS)

    # Variables de tiempo
    panel_all = pd.concat(
        [panel_all.reset_index(drop=True), add_time_feats(panel_all["ds"]).reset_index(drop=True)],
        axis=1
    )

    # Codificación categórica
    for col in ["Variedad", "Bloque", "Sector", "Gama"]:
        if col in panel_all.columns:
            panel_all = add_code_column(panel_all, col)

    # Features base
    feature_cols = [
        "yhat_prophet", "trend", "weekly", "yearly", CANT_COL,
        "Dia_Sin", "Dia_Cos", "Sem_Sin", "Sem_Cos", "Mes", "DiaSemana", "EsFinSemana",
        "Variedad_Code", "Bloque_Code"
    ]

    if "Sector_Code" in panel_all.columns:
        feature_cols.append("Sector_Code")
    if "Gama_Code" in panel_all.columns:
        feature_cols.append("Gama_Code")
    if PLANTS_COL in panel_all.columns and PROPHET_HYB_PARAMS.get("use_plants", True):
        feature_cols.append(PLANTS_COL)

    # Features derivados del target y de Cantidad
    feature_cols += [c for c in panel_all.columns if c.startswith("y_Lag_") or c.startswith("y_RollMean_")]
    feature_cols += [c for c in panel_all.columns if c.startswith(f"{CANT_COL}_Lag_") or c.startswith(f"{CANT_COL}_RollMean_")]

    # Conjunto histórico para entrenamiento del modelo de residuales
    train_resid = panel_all[(panel_all["is_future"] == 0)].copy()
    train_resid = train_resid.dropna(subset=["resid"]).copy()
    train_resid = train_resid.dropna(subset=feature_cols).copy()

    if len(train_resid) >= MIN_ROWS_GLOBAL_XGB:
        train_resid = train_resid.sort_values("ds").reset_index(drop=True)

        cutoff = train_resid["ds"].max() - pd.Timedelta(days=TEST_DAYS_RESID)
        tr = train_resid[train_resid["ds"] <= cutoff].copy()
        va = train_resid[train_resid["ds"] > cutoff].copy()

        if len(tr) >= 100 and len(va) >= 30:
            Xtr = tr[feature_cols].to_numpy()
            ytr = tr["resid"].to_numpy().astype(float)

            Xva = va[feature_cols].to_numpy()
            yva = va["resid"].to_numpy().astype(float)

            # Winsorización simple de residuales extremos
            lo, hi = np.percentile(ytr, [1, 99])
            ytr = np.clip(ytr, lo, hi)
            yva = np.clip(yva, lo, hi)

            xgb_model = XGBRegressor(**XGB_RESID_PARAMS)

            try:
                xgb_model.fit(
                    Xtr, ytr,
                    eval_set=[(Xva, yva)],
                    verbose=False
                )
            except TypeError:
                xgb_model.fit(Xtr, ytr)

            global_xgb_ready = True
            print(f"\n✅ XGB global entrenado con {len(train_resid):,} filas históricas de residuales")
            print(f"✅ Número de features XGB residual: {len(feature_cols)}")
        else:
            print("\n⚠️ No hubo suficiente split temporal para XGB global. Se usará solo Prophet.")
    else:
        print("\n⚠️ Muy pocas filas para XGB global. Se usará solo Prophet.")


# =============================================================================
# 10) PREDICCIÓN FINAL t+1
# =============================================================================

model_rows = []

if len(panel_all) > 0:
    future_rows = panel_all[panel_all["is_future"] == 1].copy()

    if global_xgb_ready:
        future_ok = future_rows.dropna(subset=feature_cols).copy()
        future_bad = future_rows[future_rows[feature_cols].isna().any(axis=1)].copy()

        if len(future_ok) > 0:
            resid_hat = xgb_model.predict(future_ok[feature_cols].to_numpy())
            future_ok["resid_hat"] = resid_hat.astype(float)

            # Mezcla entre la corrección Prophet+XGB y la predicción base de Prophet
            future_ok["y_final"] = (
                BLEND_CORRECTED * (future_ok["yhat_prophet"] + future_ok["resid_hat"])
                + BLEND_PROPHET * future_ok["yhat_prophet"]
            )
            future_ok["Metodo"] = "PROPHET+XGB_GLOBAL"

        if len(future_bad) > 0:
            future_bad["y_final"] = future_bad["yhat_prophet"]
            future_bad["Metodo"] = "PROPHET(XGB_missing_feats)"

        future_pred = pd.concat([future_ok, future_bad], ignore_index=True)
    else:
        future_pred = future_rows.copy()
        future_pred["y_final"] = future_pred["yhat_prophet"]
        future_pred["Metodo"] = "PROPHET"

    for _, r in future_pred.iterrows():
        model_rows.append({
            "Variedad": r["Variedad"],
            "Bloque": r["Bloque"],
            "Sector": r["Sector"],
            "Gama": r["Gama"],
            "Color": r["Color"],
            "Finca": r["Finca"],
            "Pais": r["Pais"],
            "Fecha_Prediccion": next_date_global.strftime("%Y-%m-%d"),
            "Metodo": r["Metodo"],
            "Cosecha_Predicha": safe_int_round(r["y_final"])
        })

# Se combinan las predicciones modeladas con las salidas fallback
rows = model_rows + fallback_rows
df_pred = pd.DataFrame(rows)

if len(df_pred) == 0:
    print("⚠️ No se generaron predicciones.")
    print(f"Saltados por no llegar a last_date_global={last_date_global.date()}: {skipped_not_current}")
    raise SystemExit()

df_pred["Cosecha_Predicha"] = df_pred["Cosecha_Predicha"].apply(safe_int_round).astype("int64")

# =============================================================================
# 11) AGRUPACIÓN ROJO VS COLOR
# =============================================================================

df_pred["ColorGrupo"] = build_color_group(df_pred)

# =============================================================================
# 12) RESÚMENES
# =============================================================================

pred_sector = resumen_rojo_color(df_pred, "Sector")
pred_gama   = resumen_rojo_color(df_pred, "Gama")

tot_rojo  = int(df_pred.loc[df_pred["ColorGrupo"] == "Rojo",  "Cosecha_Predicha"].sum())
tot_color = int(df_pred.loc[df_pred["ColorGrupo"] == "Color", "Cosecha_Predicha"].sum())
tot_all   = int(df_pred["Cosecha_Predicha"].sum())

df_totales = pd.DataFrame([{
    "Rojo_total": tot_rojo,
    "Color_total": tot_color,
    "Total": tot_all
}])

# =============================================================================
# 13) IMPRESIÓN DE RESULTADOS
# =============================================================================

pretty_print_df("Detalle t+1 (Variedad-Bloque) - primeras 25 filas", df_pred.head(25), max_rows=30)

print(f"\n📈 TOTAL COSECHA PREDICHA {next_date_global.date()}: {tot_all:,} | Rojo={tot_rojo:,} | Color={tot_color:,}")
print(f"Grupos predichos: {len(df_pred):,} | Saltados (no llegan a {last_date_global.date()}): {skipped_not_current:,}")
print(f"FALLBACK Prophet error: {prophet_error_count:,}")

pretty_print_df("📊 Resumen por Sector (Rojo | Color | Total)", pred_sector, max_rows=200)
pretty_print_df("📊 Resumen por Gama (Rojo | Color | Total)", pred_gama, max_rows=200)
pretty_print_df("📊 Totales globales (Rojo_total | Color_total | Total)", df_totales, max_rows=10)

# =============================================================================
# 14) EXPORTACIÓN A EXCEL
# =============================================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_XLSX = os.path.join(OUTPUT_DIR, f"Detalle_{next_date_global.strftime('%Y-%m-%d')}.xlsx")

try:
    with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
        df_pred.to_excel(writer, index=False, sheet_name="Detalle_t1")
        pred_sector.to_excel(writer, index=False, sheet_name="Resumen_Sector")
        pred_gama.to_excel(writer, index=False, sheet_name="Resumen_Gama")
        df_totales.to_excel(writer, index=False, sheet_name="Totales_Globales")
        df_hyper_loaded.to_excel(writer, index=False, sheet_name="HYPERPARAMS_SOURCE")

    print(f"\n💾 Archivo guardado: {OUTPUT_XLSX}")

except PermissionError as e:
    print(f"⚠️ PermissionError en OneDrive: {e}")

    OUTPUT_DIR2 = r"C:\Temp\BDENVIO"
    os.makedirs(OUTPUT_DIR2, exist_ok=True)
    OUTPUT_XLSX2 = os.path.join(OUTPUT_DIR2, f"Detalle_{next_date_global.strftime('%Y-%m-%d')}.xlsx")

    with pd.ExcelWriter(OUTPUT_XLSX2, engine="openpyxl") as writer:
        df_pred.to_excel(writer, index=False, sheet_name="Detalle_t1")
        pred_sector.to_excel(writer, index=False, sheet_name="Resumen_Sector")
        pred_gama.to_excel(writer, index=False, sheet_name="Resumen_Gama")
        df_totales.to_excel(writer, index=False, sheet_name="Totales_Globales")
        df_hyper_loaded.to_excel(writer, index=False, sheet_name="HYPERPARAMS_SOURCE")

    print(f"\n💾 Archivo guardado (fallback): {OUTPUT_XLSX2}")

print("\n✅ Listo. Queda en memoria: df_pred, pred_sector, pred_gama, df_totales")


✅ XGB global entrenado con 228,776 filas históricas de residuales
✅ Número de features XGB residual: 29

Detalle t+1 (Variedad-Bloque) - primeras 25 filas
-------------------------------------------------
       Variedad Bloque    Sector    Gama Color   Finca Pais Fecha_Prediccion             Metodo  Cosecha_Predicha ColorGrupo
ABSOLUT IN PINK     13  SECTOR 2 NOVELTY       BETANIA            2026-04-24 PROPHET+XGB_GLOBAL               386      Color
       AMATISTA     13  SECTOR 2    PINK       BETANIA            2026-04-24 PROPHET+XGB_GLOBAL               280      Color
          AMBER      8  SECTOR 1    FALL       BETANIA            2026-04-24 PROPHET+XGB_GLOBAL               662      Color
          AMBER      9  SECTOR 1    FALL       BETANIA            2026-04-24 PROPHET+XGB_GLOBAL              2473      Color
         ATOMIC   AG19    AGRADO    FALL       BETANIA            2026-04-24 PROPHET+XGB_GLOBAL              1766      Color
         ATOMIC    AG3    AGRADO    FALL    

## Cierre del proceso

Este cuaderno implementa un esquema híbrido de proyección que combina:

- la capacidad de Prophet para capturar la dinámica individual de cada grupo,
- y la capacidad de XGBoost para aprender patrones globales en los residuales del sistema.

La salida final entrega una predicción t+1 operativa por grupo, junto con resúmenes por sector, por gama y por grupos de color, manteniendo además trazabilidad sobre la fuente de los hiperparámetros utilizados.

Toda la documentación fue organizada para dejar el flujo más entendible, trazable y presentable, sin alterar la lógica original del código.